# Auditoria BCBI × SAGI (Relatórios) × SAGI (MOV)

Compara os KPIs exibidos no **BCBI** (dashboard da empresa) com os dados dos relatórios do **SAGI**:
- **Relatório de Entrada sem agrupamento** (`relatorio entrada.csv`)
- **Relatório de Saída sem agrupamento** (`relatorio saida.csv`)
- **Movimentações operacionais** (`compras.csv` / `vendas.csv`)

**Divisão:** Seletiva  
**Período:** detectado automaticamente — mês corrente

---

## Como usar

1. Abra o BCBI, filtre por **Seletiva** e pelo **mês corrente**
2. Preencha o dicionário `bcbi` na **Célula 2** com os valores que você vê no dashboard
3. Execute todas as células em sequência (`Run All`)
4. Veja o **veredicto colorido** e o **detalhamento das divergências** nas últimas células

> **Tolerância padrão:** `R$ 1.000,00` — altere `TOLERANCIA` na Célula 1 se necessário.

In [ ]:
# ─── Imports e configuração ────────────────────────────────────────────────
import os
import pandas as pd
import numpy as np
from datetime import date, datetime
from IPython.display import display, HTML

# Caminhos: detecta automaticamente se o notebook roda da raiz, de 04-Notebooks/ ou de subpastas.
_cwd = os.getcwd()
if os.path.isdir(os.path.join(_cwd, '02-Referencias')):
    WORKSPACE = _cwd                                    # CWD = raiz do workspace
elif os.path.isdir(os.path.join(_cwd, '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..'))  # CWD = 04-Notebooks/
elif os.path.isdir(os.path.join(_cwd, '..', '..', '02-Referencias')):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, '..', '..'))  # CWD = 04-Notebooks/subpasta/
else:
    raise FileNotFoundError(f'Pasta 02-Referencias nao encontrada a partir de: {_cwd}')

REFS            = os.path.join(WORKSPACE, '02-Referencias')
PATH_ENTRADA_REL = os.path.join(REFS, 'relatorio entrada.csv')
PATH_SAIDA_REL   = os.path.join(REFS, 'relatorio saida.csv')
PATH_COMPRAS     = os.path.join(REFS, 'Meus Dados', 'compras.csv')
PATH_VENDAS      = os.path.join(REFS, 'Meus Dados', 'vendas.csv')

# Período: mês corrente
hoje      = datetime.now()
MES       = f'{hoje.year}-{hoje.month:02d}'   # ex: '2026-03'  (formato ODBC vencimento)
MES_LABEL = hoje.strftime('%m/%Y')            # ex: '03/2026'
MES_LANC  = hoje.strftime('/%m/%Y')           # ex: '/03/2026' (formato coluna lancamento DD/MM/YYYY)

# Tolerância de diferença aceitável (R$)
TOLERANCIA = 1_000.00

print(f'Período de análise : {MES_LABEL}  ({MES}-xx)')
print(f'Tolerância         : R$ {TOLERANCIA:,.2f}')
print(f'REL ENTRADA SAGI   : {PATH_ENTRADA_REL}')
print(f'REL SAÍDA SAGI     : {PATH_SAIDA_REL}')
print(f'MOV COMPRAS        : {PATH_COMPRAS}')
print(f'MOV VENDAS         : {PATH_VENDAS}')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#   PREENCHA AQUI OS VALORES DO BCBI
#   (Dashboard → Seletiva → mês corrente)
#
#   Linha 1 do dashboard:  RECEITAS TOTAIS | COMPRAS | CUSTOS TOTAIS | ESTOQUE | RESULTADO
#   Linha 2 do dashboard:  VENDAS DE SUCATA (R$ e KG) | COMPRAS DE SUCATA (R$ e KG)
# ══════════════════════════════════════════════════════════════════════════════

bcbi = {
    # ── Linha 1 ───────────────────────────────────────────────────────────────
    'receitas_totais_rs':    8_999_716.92,   # card RECEITAS TOTAIS
    'compras_financeiro_rs': 3_098_798.17,   # card COMPRAS (V. Bruto)
    'custos_totais_rs':      2_230_288.38,   # card CUSTOS TOTAIS
    'estoque_rs':            3_123_843.20,   # card ESTOQUE
    'resultado_rs':            546_787.17,   # card RESULTADO

    # ── Linha 2 ───────────────────────────────────────────────────────────────
    'vendas_sucata_rs':      8_947_580.66,   # card VENDAS DE SUCATA R$ (usar valor bruto)
    'vendas_sucata_kg':      5_781_080,      # card VENDAS DE SUCATA KG
    'compras_sucata_rs':     3_095_048.57,   # card COMPRAS DE SUCATA R$
    'compras_sucata_kg':     3_774_441,      # card COMPRAS DE SUCATA KG
}

print(f"BCBI preenchido em {hoje.strftime('%d/%m/%Y')} — {len(bcbi)} métricas")

In [ ]:
# ─── Carga e filtragem dos dados ────────────────────────────────────────────

def _to_float(series):
    """Converte coluna numérica com vírgula decimal (padrão BR) para float."""
    return pd.to_numeric(
        series.astype(str).str.strip().str.replace(',', '.', regex=False),
        errors='coerce'
    )

# ── Relatórios SAGI (Entrada/Saída sem agrupamento) ─────────────────────────
def _parse_relatorio_sagi_sem_agrupamento(path_csv, tipo):
    """Extrai linhas de detalhe do relatório textual exportado pelo SAGI."""
    rows = []
    with open(path_csv, 'r', encoding='latin-1', errors='ignore') as f:
        for raw in f:
            line = raw.rstrip('\n')
            if not line or ';' not in line:
                continue

            parts = [p.strip() for p in line.split(';')]
            if not parts or not parts[0].isdigit():
                continue

            # O layout vem com muitos delimitadores vazios; usar a sequência não vazia
            vals = [p for p in parts if p]
            if len(vals) < 10:
                continue

            # Estrutura esperada da sequência principal:
            # [boleto, nota, data, pessoa, quantidade, preco, total, total_icms, placa, produto, ...]
            idx_data = next((i for i, v in enumerate(vals) if len(v) == 10 and v[2] == '/' and v[5] == '/'), None)
            if idx_data is None or idx_data + 8 >= len(vals):
                continue

            boleto = vals[0]
            data = vals[idx_data]
            pessoa = vals[idx_data + 1]
            quantidade = vals[idx_data + 2]
            valor_total = vals[idx_data + 4]
            produto = vals[idx_data + 7]

            rows.append({
                'tipo': tipo,
                'boleto': boleto,
                'data': data,
                'pessoa': pessoa,
                'produto': produto,
                'quantidade_kg': quantidade,
                'valor_total': valor_total,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out

    out['data'] = pd.to_datetime(out['data'], format='%d/%m/%Y', errors='coerce')

    out['quantidade_kg'] = (
        out['quantidade_kg'].astype(str)
        .str.replace(r'[^0-9,.-]', '', regex=True)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    out['valor_total'] = (
        out['valor_total'].astype(str)
        .str.replace(r'[^0-9,.-]', '', regex=True)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
    )

    out['quantidade_kg'] = pd.to_numeric(out['quantidade_kg'], errors='coerce')
    out['valor_total'] = pd.to_numeric(out['valor_total'], errors='coerce')

    # O relatório pode vir consolidando múltiplas filiais sem coluna explícita de filial.
    out['filial'] = 'Consolidado SAGI'
    return out

print('Carregando relatório SAGI de entrada ...')
df_entrada_rel = _parse_relatorio_sagi_sem_agrupamento(PATH_ENTRADA_REL, 'ENTRADA')
df_entrada_rel = df_entrada_rel[df_entrada_rel['data'].dt.strftime('%Y-%m').eq(MES)].copy()

# Compras de sucata no relatório de entrada: usa descrição do produto
_prod_ent = df_entrada_rel['produto'].astype(str).str.upper()
_mask_compra_suc = _prod_ent.str.contains('SUCATA', na=False)
df_entrada_rel_sucata = df_entrada_rel[_mask_compra_suc].copy()

print(f'  Registros Entrada (mês {MES_LABEL})              : {len(df_entrada_rel):,}')
print(f'  Registros Compras Sucata (mês {MES_LABEL})       : {len(df_entrada_rel_sucata):,}')

print('\nCarregando relatório SAGI de saída ...')
df_saida_rel = _parse_relatorio_sagi_sem_agrupamento(PATH_SAIDA_REL, 'SAIDA')
df_saida_rel = df_saida_rel[df_saida_rel['data'].dt.strftime('%Y-%m').eq(MES)].copy()

# Vendas de sucata: apenas FERRO (1) e MAT PROCESSADO (24)
# No relatório sem agrupamento não há cod_cat_pro, então usamos descrição do produto.
_prod = df_saida_rel['produto'].astype(str).str.upper()
_mask_ferro = _prod.str.contains('FERRO', na=False)
_mask_mat_proc = _prod.str.contains('PROCESS', na=False)
_mask_vendas_suc = _mask_ferro | _mask_mat_proc

df_saida_rel_sucata = df_saida_rel[_mask_vendas_suc].copy()

print(f'  Registros Saída (mês {MES_LABEL})                  : {len(df_saida_rel):,}')
print(f'  Registros Vendas Sucata [FERRO+MAT PROC] ({MES_LABEL}): {len(df_saida_rel_sucata):,}')

# ── SAGI MOV COMPRAS ─────────────────────────────────────────────────────────
print('\nCarregando compras.csv ...')
df_compras = pd.read_csv(PATH_COMPRAS, sep=';', encoding='latin-1', dtype=str)
df_compras['valor_total']   = _to_float(df_compras['valor_total'])
df_compras['n_qtde_fat_kg'] = _to_float(df_compras['n_qtde_fat_kg'])
df_compras = df_compras[df_compras['data'].str.startswith(MES, na=False)].copy()
df_compras = df_compras[df_compras['c_tipo_movimento'] == 'ENTRADA'].copy()
# Apenas filiais Seletiva (G3S Dourados, Londrina, Maringá, Prudente, CG, Cidade Alta, Assis)
df_compras = df_compras[df_compras['c_filial_carregamento'].str.startswith('G3S', na=False)].copy()

# O BCBI tem dois cards de compra com filtros diferentes — dois DataFrames:
# ① "Valor Compras" (card Compras Financeiro): FERRO(1) + MAT PROCESSADO(24)
_CAT_COMPRAS_FIN = ['1', '24']
df_compras_fin = df_compras[df_compras['cod_cat_pro'].isin(_CAT_COMPRAS_FIN)].copy()
print(f'  Compras Financeiro (FERRO+MAT PROC) : {len(df_compras_fin):,} registros')
print(f'    Categorias: {df_compras_fin["des_cat_pro"].unique().tolist()}')

# ② "Valor Compras de Sucata" (card Compras Sucata): Ferro(1), Escolha(4), Mat.Fino(20), Plastico/Papel(8), Eletronicos(4925)
_CAT_COMPRAS_SUC = ['1', '4', '8', '20', '4925']
df_compras_suc = df_compras[df_compras['cod_cat_pro'].isin(_CAT_COMPRAS_SUC)].copy()
print(f'  Compras Sucata (5 categorias)        : {len(df_compras_suc):,} registros')
print(f'    Categorias: {df_compras_suc["des_cat_pro"].unique().tolist()}')

# ── SAGI MOV VENDAS ──────────────────────────────────────────────────────────
print('\nCarregando vendas.csv ...')
df_vendas = pd.read_csv(PATH_VENDAS, sep=';', encoding='latin-1', dtype=str)
df_vendas['valor_total']   = _to_float(df_vendas['valor_total'])
df_vendas['n_qtde_fat_kg'] = _to_float(df_vendas['n_qtde_fat_kg'])
df_vendas = df_vendas[df_vendas['data'].str.startswith(MES, na=False)].copy()
# SAIDA + DEV-SAIDA (devoluções têm valores negativos — entram na soma líquida)
df_vendas = df_vendas[df_vendas['c_tipo_movimento'].isin(['SAIDA', 'DEV-SAIDA'])].copy()
df_vendas = df_vendas[df_vendas['c_filial_carregamento'].str.startswith('G3S', na=False)].copy()
# Apenas categorias de vendas de sucata: FERRO (1) e MAT PROCESSADO (24)
df_vendas = df_vendas[df_vendas['cod_cat_pro'].isin(['1', '24'])].copy()
print(f'  Registros Vendas Seletiva  : {len(df_vendas):,}')
print(f'  Categorias: {df_vendas["des_cat_pro"].unique().tolist()}')

print('\nDados carregados com sucesso.')

In [ ]:
# ─── Métricas SAGI (Relatórios Entrada/Saída sem agrupamento) ───────────────

# Mantemos os nomes de variáveis para não quebrar as células seguintes.
# Agora a origem é: relatórios de Entrada/Saída gerados no SAGI.
odbc_receitas_totais    = df_saida_rel['valor_total'].sum()
odbc_vendas_sucata_rs   = df_saida_rel_sucata['valor_total'].sum()
odbc_compras_fin_rs     = df_entrada_rel['valor_total'].sum()
odbc_compras_sucata_rs  = df_entrada_rel_sucata['valor_total'].sum()

# Pesos vindos diretamente dos relatórios de Entrada/Saída
odbc_vendas_sucata_kg   = df_saida_rel_sucata['quantidade_kg'].sum()
odbc_compras_sucata_kg  = df_entrada_rel_sucata['quantidade_kg'].sum()

# Esses KPIs dependiam da estrutura contábil do base.csv (plano de contas/CC).
# Nos relatórios sem agrupamento eles não existem de forma equivalente.
odbc_custos_totais    = np.nan
odbc_resultado_semest = np.nan

print(f'Métricas SAGI (Relatórios) — Seletiva — {MES_LABEL}')
print(f'  Receitas Totais (Saída)       : R$ {odbc_receitas_totais:>15,.2f}')
print(f'  Vendas Sucata (Saída)         : R$ {odbc_vendas_sucata_rs:>15,.2f}')
print(f'  Vendas Sucata (Saída)         :    {odbc_vendas_sucata_kg:>15,.0f} kg')
print(f'  Compras Financeiro (Entrada)  : R$ {odbc_compras_fin_rs:>15,.2f}')
print(f'  Compras Sucata (Entrada)      : R$ {odbc_compras_sucata_rs:>15,.2f}')
print(f'  Compras Sucata (Entrada)      :    {odbc_compras_sucata_kg:>15,.0f} kg')
print('  Custos Totais                 : não disponível neste relatório')
print('  Resultado                     : não disponível neste relatório')

In [ ]:
# ─── Métricas SAGI (movimentações operacionais) ──────────────────────────────

# ① Compras Financeiro — FERRO + MAT PROCESSADO (card "Valor Compras" do BCBI)
sagi_compras_fin_rs = df_compras_fin['valor_total'].sum()
sagi_compras_fin_kg = df_compras_fin['n_qtde_fat_kg'].sum()

# ② Compras Sucata — 5 categorias (card "Valor Compras de Sucata" do BCBI)
sagi_compras_suc_rs = df_compras_suc['valor_total'].sum()
sagi_compras_suc_kg = df_compras_suc['n_qtde_fat_kg'].sum()

sagi_vendas_rs  = df_vendas['valor_total'].sum()
sagi_vendas_kg  = df_vendas['n_qtde_fat_kg'].sum()

cmp_fin_rkg = sagi_compras_fin_rs / sagi_compras_fin_kg if sagi_compras_fin_kg else 0
cmp_suc_rkg = sagi_compras_suc_rs / sagi_compras_suc_kg if sagi_compras_suc_kg else 0
vnd_rkg     = sagi_vendas_rs       / sagi_vendas_kg      if sagi_vendas_kg      else 0

print(f'Métricas SAGI — Seletiva — {MES_LABEL}')
print(f'  Compras Financeiro (R$) : R$ {sagi_compras_fin_rs:>15,.2f}   ({cmp_fin_rkg:.2f} R$/kg)  ← FERRO+MAT PROC')
print(f'  Compras Financeiro (KG) :    {sagi_compras_fin_kg:>15,.0f} kg')
print(f'  Compras Sucata     (R$) : R$ {sagi_compras_suc_rs:>15,.2f}   ({cmp_suc_rkg:.2f} R$/kg)  ← 5 categorias')
print(f'  Compras Sucata     (KG) :    {sagi_compras_suc_kg:>15,.0f} kg')
print(f'  Vendas  Sucata     (R$) : R$ {sagi_vendas_rs:>15,.2f}   ({vnd_rkg:.2f} R$/kg)')
print(f'  Vendas  Sucata     (KG) :    {sagi_vendas_kg:>15,.0f} kg')

In [ ]:
# ─── Tabela comparativa e veredicto ─────────────────────────────────────────

_NA = float('nan')

# ── Funções de formatação ─────────────────────────────────────────────────────
def _fmtv(v, kg=False):
    if pd.isna(v): return '—'
    return f'{v:,.0f} kg' if kg else f'R$ {v:,.2f}'

def _fmtd(v):
    if pd.isna(v): return '—'
    return f'{v:+,.2f}'

def _fmtp(diff, base):
    if pd.isna(diff) or pd.isna(base) or base == 0: return '—'
    return f'{diff / base * 100:+.2f}%'

# ── Severidade e paleta de cores ──────────────────────────────────────────────
_RANK = {'error': 3, 'warn': 2, 'ok': 1, 'none': 0}

def _severity(v):
    """Severidade pelo valor absoluto da diferença."""
    if pd.isna(v):           return 'none'
    if abs(v) <= TOLERANCIA: return 'ok'
    if abs(v) <= 10_000:     return 'warn'
    return                           'error'

def _severity_pct(diff, base, pct_ok=0.01):
    """Severidade considerando também o percentual: se |Δ/base| < pct_ok → OK."""
    if pd.isna(diff): return 'none'
    if not (pd.isna(base) or base == 0) and abs(diff / base) < pct_ok:
        return 'ok'
    return _severity(diff)

_ICON = {'error': '❌', 'warn': '⚠️', 'ok': '✅', 'none': '∅'}

def _cor_delta(v):
    """Cor da célula Δ: verde sólido / âmbar / vermelho sólido / cinza (sem comparação)."""
    if pd.isna(v):
        return 'background-color:#e9ecef;color:#6c757d'
    if abs(v) <= TOLERANCIA:
        return 'background-color:#198754;color:#ffffff;font-weight:bold'
    if abs(v) <= 10_000:
        return 'background-color:#fd7e14;color:#ffffff;font-weight:bold'
    return     'background-color:#dc3545;color:#ffffff;font-weight:bold'

def _cor_pct(v):
    """Cor da célula % com base no valor numérico."""
    if v == '—': return 'background-color:#e9ecef;color:#6c757d'
    try:
        n = abs(float(v.replace('%','').replace('+','').replace('-','')))
        if n <= 0.5:  return 'color:#198754;font-weight:bold'
        if n <= 5.0:  return 'color:#fd7e14;font-weight:bold'
        return               'color:#dc3545;font-weight:bold'
    except:
        return ''

def _cor_empty(v):
    """Cinza para células sem fonte de comparação."""
    if v == '—': return 'background-color:#e9ecef;color:#6c757d'
    return ''

# ── Definição das métricas (somente BCBI vs Relatórios SAGI) ─────────────────
linhas = [
    ('Receitas Totais (R$)',    False, bcbi['receitas_totais_rs'],    odbc_receitas_totais),
    ('Vendas Sucata (R$)',      False, bcbi['vendas_sucata_rs'],      odbc_vendas_sucata_rs),
    ('Vendas Sucata (KG)',      True,  bcbi['vendas_sucata_kg'],      odbc_vendas_sucata_kg),
    ('Compras Financeiro (R$)', False, bcbi['compras_financeiro_rs'], odbc_compras_fin_rs),
    ('Compras Sucata (R$)',     False, bcbi['compras_sucata_rs'],     odbc_compras_sucata_rs),
    ('Compras Sucata (KG)',     True,  bcbi['compras_sucata_kg'],     odbc_compras_sucata_kg),
    ('Custos Totais (R$)',      False, bcbi['custos_totais_rs'],      odbc_custos_totais),
    ('Estoque (R$)',            False, bcbi['estoque_rs'],            _NA),
    ('Resultado (R$)',          False, bcbi['resultado_rs'],          odbc_resultado_semest),
]

rows        = []
sev_bo_list = []

for label, kg, b, o in linhas:
    d_bo   = b - o if not (pd.isna(b) or pd.isna(o)) else _NA
    # Regra de status: OK quando |% Relatório| < 1%
    sev_bo = _severity_pct(d_bo, o, pct_ok=0.01)
    sev_bo_list.append(sev_bo)
    rows.append({
        'Status':                      _ICON[sev_bo],
        'Métrica':                     label,
        'BCBI':                        _fmtv(b, kg),
        'SAGI (Rel. Entrada/Saída)':   _fmtv(o, kg),
        'Δ BCBI-Relatório':            d_bo,
        '% Relatório':                 _fmtp(d_bo, o),
    })

# Remove as 3 últimas métricas do relatório exibido
rows = rows[:-3]
sev_bo_list = sev_bo_list[:-3]
cmp = pd.DataFrame(rows)

# ── Estilos base ──────────────────────────────────────────────────────────────
_TH  = ('background-color:#1f3864;color:#ffffff;font-weight:600;'
        'padding:9px 13px;text-align:center;white-space:nowrap;font-size:13px')
_TD  = 'padding:7px 12px;border:1px solid #dee2e6;font-size:13px;text-align:right'
_TDL = 'padding:7px 12px;border:1px solid #dee2e6;font-size:13px;text-align:left;font-weight:500'
_TDC = 'padding:7px 8px;border:1px solid #dee2e6;font-size:15px;text-align:center'

styled = (
    cmp.style
       .format({'Δ BCBI-Relatório': _fmtd}, na_rep='—')
       .set_table_styles([
           {'selector': 'table',            'props': 'border-collapse:collapse;width:100%;font-family:system-ui,-apple-system,sans-serif'},
           {'selector': 'th',               'props': _TH},
           {'selector': 'td',               'props': _TD},
           {'selector': 'td:nth-child(1)',   'props': _TDC},           # Status — ícone centralizado
           {'selector': 'td:nth-child(2)',   'props': _TDL},           # Métrica — texto à esquerda
           {'selector': 'td:nth-child(3)',   'props': _TD + ';font-weight:600'},  # BCBI — negrito
           {'selector': 'tr:nth-child(even)', 'props': 'background-color:#f8f9fa'},
           {'selector': 'tr:hover td',       'props': 'filter:brightness(0.94)'},
       ])
)

# Colorir Δ (numérico: NaN → cinza, ok → verde, warn → âmbar, error → vermelho)
try:
    styled = styled.map(_cor_delta, subset=['Δ BCBI-Relatório'])
    styled = styled.map(_cor_pct,   subset=['% Relatório'])
    styled = styled.map(_cor_empty, subset=['SAGI (Rel. Entrada/Saída)'])
except AttributeError:
    styled = styled.applymap(_cor_delta, subset=['Δ BCBI-Relatório'])
    styled = styled.applymap(_cor_pct,   subset=['% Relatório'])
    styled = styled.applymap(_cor_empty, subset=['SAGI (Rel. Entrada/Saída)'])

# Cor de texto escura nas linhas com fundo claro (índices 1, 3, 5, 7 — tr:nth-child(even))
_COLS_TEXTO = ['Métrica', 'BCBI', 'SAGI (Rel. Entrada/Saída)']

def _cor_texto_fundo_claro(df):
    styles = pd.DataFrame('', index=df.index, columns=df.columns)
    for idx in df.index:
        if idx % 2 == 1:
            for col in _COLS_TEXTO:
                styles.at[idx, col] = 'color:#181818'
    return styles

styled = styled.apply(_cor_texto_fundo_claro, axis=None)

display(styled)

# ── Contadores por severidade ─────────────────────────────────────────────────
n_err  = cmp['Status'].eq(_ICON['error']).sum()
n_warn = cmp['Status'].eq(_ICON['warn']).sum()
n_ok   = cmp['Status'].eq(_ICON['ok']).sum()
n_none = cmp['Status'].eq(_ICON['none']).sum()

summary = (
    f'<span style="color:#198754;font-weight:bold">✅ {n_ok} OK</span>&nbsp;&nbsp;|&nbsp;&nbsp;'
    f'<span style="color:#fd7e14;font-weight:bold">⚠️ {n_warn} Atenção</span>&nbsp;&nbsp;|&nbsp;&nbsp;'
    f'<span style="color:#dc3545;font-weight:bold">❌ {n_err} Divergência</span>&nbsp;&nbsp;|&nbsp;&nbsp;'
    f'<span style="color:#6c757d">∅ {n_none} Sem fonte</span>'
)

# ── Veredicto ─────────────────────────────────────────────────────────────────
div_bo = cmp[[s != 'ok' and s != 'none' for s in sev_bo_list]]
total_div = len(div_bo)
now_str   = hoje.strftime('%d/%m/%Y %H:%M')

if total_div == 0:
    box = ('background:linear-gradient(120deg,#d4edda,#c3e6cb);color:#155724;'
           'padding:16px 20px;border-radius:10px;border-left:5px solid #198754;'
           'font-size:1.05em;font-weight:bold;margin-top:14px')
    display(HTML(
        f'<div style="{box}">'
        f'✅  AUDITORIA OK — Todos os valores batem (tolerância R$ {TOLERANCIA:,.0f})&nbsp;&nbsp;|&nbsp;&nbsp;{now_str}'
        f'<div style="font-weight:normal;font-size:0.88em;margin-top:6px">{summary}</div>'
        f'</div>'
    ))
else:
    box = ('background:linear-gradient(120deg,#f8d7da,#f5c6cb);color:#721c24;'
           'padding:16px 20px;border-radius:10px;border-left:5px solid #dc3545;'
           'font-size:1.05em;font-weight:bold;margin-top:14px')
    items_html = ''
    for _, row in div_bo.iterrows():
        items_html += (f'<li><b>{row["Métrica"]}</b> (BCBI vs Relatório SAGI): '
                       f'Δ = {_fmtd(row["Δ BCBI-Relatório"])} &nbsp; {row["% Relatório"]}</li>')
    display(HTML(
        f'<div style="{box}">'
        f'⚠️  DIVERGÊNCIAS ENCONTRADAS: {total_div} métrica(s)&nbsp;&nbsp;|&nbsp;&nbsp;{now_str}'
        f'<ul style="font-weight:normal;margin-top:8px;margin-bottom:8px">{items_html}</ul>'
        f'<div style="font-weight:normal;font-size:0.88em;border-top:1px solid #f5c6cb;padding-top:6px">{summary}</div>'
        f'</div>'
    ))